In [1]:
from anndata import read_h5ad

In [2]:
from os.path import join

In [3]:
adata = read_h5ad(join("data", "intermediate", "combined.subclass_l1.specimen.sum.pdata.h5ad"))

/Users/mkeller/research/dbmi/vitessce/compasce-degs/.venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1787: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [4]:
adata

AnnData object with n_obs × n_vars = 2916 × 36588
    obs: 'library_id', 'nCount_RNA', 'nFeature_RNA', 'percent.er', 'percent.mt', 'experiment_id', 'specimen', 'patient', 'region', 'percent.cortex', 'percent.medulla', 'subclass_l3', 'subclass_l2', 'subclass_l1', 'class', 'UMAP_1', 'UMAP_2', 'Participant ID', 'Tissue Source', 'Protocol', 'Sample Type', 'Enrollment Category', 'Primary Adjudicated Category', 'Sex', 'Age (Years) (Binned)', 'Race', 'KDIGO Stage', 'Baseline eGFR (ml per min per 1.73m2)', 'Baseline eGFR (ml per min per 1.73m2) (Binned)', 'Proteinuria (mg) (Binned)', 'A1c (%) (Binned)', 'Albuminuria (mg) (Binned)', 'Diabetes History', 'Diabetes Duration (Years)', 'Hypertension History', 'Hypertension Duration (Years)', 'On RAAS Blockade', 'AdjudicatedCategory', 'EnrollmentCategory', 'num_cells_orig'
    var: 'gene_symbol', 'ensembl_id'

In [5]:
adata.obs["subclass_l1"].unique().tolist()

['Ad',
 'ATL',
 'CNT',
 'DCT',
 'DTL',
 'EC',
 'FIB',
 'IC',
 'Lymphoid',
 'Myeloid',
 'NEU',
 'PapE',
 'PC',
 'PEC',
 'POD',
 'PT',
 'TAL',
 'VSM/P']

In [6]:
adata.obs["specimen"].unique()

['18-142-3-M2', '18-162-2-M2', '18-312-2-M2', '446_B1', '446_B3', ..., 'S-2305-006657_D1_N1', 'S-2306-001149_D1_N1', 'S-2306-012843_D1_N1', 'S-2307-011657_D1_N1', 'S-2307-011845_D1_N1']
Length: 162
Categories (162, object): ['18-142-3-M2', '18-162-2-M2', '18-312-2-M2', '446_B1', ..., 'S-2306-001149_D1_N1', 'S-2306-012843_D1_N1', 'S-2307-011657_D1_N1', 'S-2307-011845_D1_N1']

In [7]:
adata.obs["num_cells_orig"].sum()

575964

In [ ]:
# TODO: filtering based on num samples and num cells

In [11]:
import pertpy as pt

/Users/mkeller/research/dbmi/vitessce/compasce-degs/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
pdata = adata

In [15]:
pdata.shape

(2916, 36588)

In [10]:
cell_type_col = "subclass_l1"

In [12]:
cell_types = pdata.obs[cell_type_col].unique().tolist()

In [14]:
for cell_type in cell_types:
    # Cell type vs rest
    pdata.obs["is_cell_type"] = pdata.obs[cell_type_col].apply(lambda x: cell_type if x == cell_type else "rest")

    # For cell type vs. rest, we use a design such as "~subclass_l1"
    # Reference: https://hbctraining.github.io/DGE_workshop/lessons/04_DGE_DESeq2_analysis.html
    pds2 = pt.tl.PyDESeq2(adata=pdata, design=f"~is_cell_type")
    pds2.fit()

    df = pds2.test_contrasts(pds2.contrast(column="is_cell_type", baseline="rest", group_to_compare=cell_type))

Fitting size factors...
/Users/mkeller/research/dbmi/vitessce/compasce-degs/.venv/lib/python3.12/site-packages/pydeseq2/dds.py:532: UserWarning: Every gene contains at least one zero, cannot compute log geometric means. Switching to iterative mode.
  self.fit_size_factors(


Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 7.48 seconds.

Fitting MAP dispersions...
... done in 8.55 seconds.



KeyboardInterrupt: 